# Phase 11 (FINAL) — Conformalized Quantile Regression (CQR) for EF Intervals

**Why this replaces the Gaussian heteroscedastic conformal approach from notebook 10:** that approach assumed the error distribution around the mean prediction is well-described by a single learned Gaussian variance, then conformal-corrected it. The resulting intervals were valid (coverage close to target) but wide (~34 EF points), because the variance head was not well-calibrated to the true residual distribution, forcing a large correction factor.

**CQR (Romano, Patterson & Candes, 2019)** instead directly learns the lower and upper quantiles of the EF distribution (5th and 95th percentile, via pinball/quantile loss) -- no Gaussian assumption. Conformal calibration then only needs a small additive correction, typically yielding tighter, *adaptive* intervals (wider for genuinely harder cases, narrower for easy ones) while keeping the same finite-sample coverage guarantee.

**Classification (HFrEF) is unchanged** from notebook 10 -- the predictive-entropy + Clopper-Pearson guaranteed-accuracy approach already produced a clean, monotonic curve and does not need replacing.

**This is the final iteration.** After this notebook's one-time test evaluation, no further changes should be made based on test results -- this becomes the paper's definitively reported model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error
from scipy import stats

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
FRAME_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/full', 'frames')
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')
PRETRAINED_BACKBONE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'simclr_pretrain_v1', 'backbone_only.pt')

assert os.path.exists(PRETRAINED_BACKBONE_PATH), 'Run 07_selfsupervised_pretraining.ipynb first.'

df = pd.read_csv(FULL_CSV)
quality_df = pd.read_csv(quality_csv_path)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
HFrEF_THRESHOLD = 40

for col in ['sharpness', 'contrast']:
    lo, hi = quality_df[col].quantile(0.02), quality_df[col].quantile(0.98)
    quality_df[col + '_norm'] = ((quality_df[col] - lo) / max(hi - lo, 1e-6)).clip(0, 1)
quality_df['quality_score'] = quality_df[['sharpness_norm', 'contrast_norm']].mean(axis=1)
df = df.merge(quality_df[[filename_col, 'quality_score']], on=filename_col, how='left')

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(FRAME_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask & df['quality_score'].notna()].reset_index(drop=True)
train_df = clean_df[clean_df['Split'].str.upper() == 'TRAIN'].reset_index(drop=True)
val_df = clean_df[clean_df['Split'].str.upper() == 'VAL'].reset_index(drop=True)
test_df = clean_df[clean_df['Split'].str.upper() == 'TEST'].reset_index(drop=True)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)} (test untouched until the final cell -- LAST TIME)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU before running this notebook.')

In [ ]:
# Reproducibility: fix random seeds (used for all runs from this version onward)
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Note: GPU kernels (cuDNN) can still be non-deterministic, so re-runs may differ slightly in the last digits.

In [ ]:
class EchoFrameDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        aug = [T.RandomHorizontalFlip()] if train else []
        self.transform = T.Compose([T.ToPILImage(), T.Resize((112, 112)), *aug, T.ToTensor(), T.Normalize(mean, std)])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        frame = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        img = self.transform(frame)
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        quality = float(row['quality_score']) if not pd.isna(row['quality_score']) else 0.5
        return img, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

BATCH_SIZE = 32
train_ds = EchoFrameDataset(train_df, FRAME_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoFrameDataset(val_df, FRAME_CACHE_DIR, filename_col, ef_col, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Model: same FiLM-conditioned backbone, now with quantile heads instead of a Gaussian variance head

In [ ]:
class QualityFiLM(nn.Module):
    def __init__(self, feat_dim=512, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 2 * feat_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.feat_dim = feat_dim

    def forward(self, quality_score):
        params = self.net(quality_score.unsqueeze(-1))
        gamma, beta = params[:, :self.feat_dim], params[:, self.feat_dim:]
        return 1.0 + gamma, beta

class EchoCQR(nn.Module):
    def __init__(self, dropout_p=0.2, pretrained_backbone_path=None):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()
        if pretrained_backbone_path:
            backbone.load_state_dict(torch.load(pretrained_backbone_path, map_location='cpu'))
        self.backbone = backbone
        self.film = QualityFiLM(feat_dim=512)
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_mean_head = nn.Linear(512, 1)
        self.ef_q_lo_head = nn.Linear(512, 1)
        self.ef_q_hi_head = nn.Linear(512, 1)
        self.cls_head = nn.Linear(512, 1)

    def forward(self, x, quality_score):
        feat = self.backbone(x)
        gamma, beta = self.film(quality_score)
        feat = gamma * feat + beta
        feat = self.dropout(feat)
        ef_mean = self.ef_mean_head(feat).squeeze(-1)
        q_lo = self.ef_q_lo_head(feat).squeeze(-1)
        q_hi = self.ef_q_hi_head(feat).squeeze(-1)
        cls_logit = self.cls_head(feat).squeeze(-1)
        return ef_mean, q_lo, q_hi, cls_logit

def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return torch.mean(torch.max(tau * diff, (tau - 1) * diff))

TAU_LO, TAU_HI = 0.05, 0.95

## Train (checkpointed, resumable)

In [ ]:
EXP_NAME = 'cqr_film_v1'
model = EchoCQR(pretrained_backbone_path=PRETRAINED_BACKBONE_PATH).to(device)
with torch.no_grad():
    mean_ef = float(train_df[ef_col].mean())
    std_ef = float(train_df[ef_col].std())
    model.ef_mean_head.bias.fill_(mean_ef); model.ef_mean_head.weight.zero_()
    model.ef_q_lo_head.bias.fill_(mean_ef - std_ef); model.ef_q_lo_head.weight.zero_()
    model.ef_q_hi_head.bias.fill_(mean_ef + std_ef); model.ef_q_hi_head.weight.zero_()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
pos_rate = (train_df[ef_col] < HFrEF_THRESHOLD).mean()
pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

STATE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', EXP_NAME, 'latest.pt')
os.makedirs(os.path.dirname(STATE_PATH), exist_ok=True)
start_epoch, resume_step = 0, 0
if os.path.exists(STATE_PATH):
    state = torch.load(STATE_PATH, map_location=device)
    model.load_state_dict(state['model_state']); optimizer.load_state_dict(state['optimizer_state'])
    start_epoch, resume_step = state['epoch'], state['step_in_epoch']
    print(f'[{EXP_NAME}] resumed from epoch={start_epoch}')

N_EPOCHS = 10
CHECKPOINT_EVERY_MINUTES = 15
last_ckpt_time = time.time()

def evaluate(model, loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for imgs, ef_true, cls_true, quality in loader:
            imgs, quality = imgs.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(imgs, quality)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_mean.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    return {'mae': mae, 'auc': auc}

for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    for step, (imgs, ef_true, cls_true, quality) in enumerate(train_loader):
        if epoch == start_epoch and step < resume_step:
            continue
        imgs, ef_true, cls_true, quality = imgs.to(device), ef_true.to(device), cls_true.to(device), quality.to(device)
        ef_mean, q_lo, q_hi, cls_logit = model(imgs, quality)
        loss = (F.l1_loss(ef_mean, ef_true)
                + pinball_loss(ef_true, q_lo, TAU_LO)
                + pinball_loss(ef_true, q_hi, TAU_HI)
                + cls_criterion(cls_logit, cls_true))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        if (time.time() - last_ckpt_time) > CHECKPOINT_EVERY_MINUTES * 60:
            torch.save({'epoch': epoch, 'step_in_epoch': step + 1, 'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict()}, STATE_PATH)
            last_ckpt_time = time.time()
    resume_step = 0
    torch.save({'epoch': epoch + 1, 'step_in_epoch': 0, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict()}, STATE_PATH)
    val_metrics = evaluate(model, val_loader)
    print(f'Epoch {epoch}: val_mae={val_metrics["mae"]:.3f} val_auc={val_metrics["auc"]:.3f}')

print('Training complete (or reached N_EPOCHS). Re-run to resume after a disconnect.')

## CQR calibration on VAL

Standard split-CQR nonconformity score: `E = max(q_lo - y, y - q_hi)`. The (1-alpha) empirical quantile of E on a calibration split gives a single additive correction `Q_hat`; the final interval is `[q_lo - Q_hat, q_hi + Q_hat]`. This keeps the shape (width, asymmetry) learned per-example by the quantile heads, only correcting overall coverage -- typically both valid and tighter/more adaptive than the Gaussian approach.

In [ ]:
def predict_all(model, loader):
    model.eval()
    ef_true_all, ef_mean_all, q_lo_all, q_hi_all, cls_true_all, cls_prob_all = [], [], [], [], [], []
    with torch.no_grad():
        for imgs, ef_true, cls_true, quality in loader:
            imgs, quality = imgs.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(imgs, quality)
            ef_true_all.extend(ef_true.numpy()); ef_mean_all.extend(ef_mean.cpu().numpy())
            q_lo_all.extend(q_lo.cpu().numpy()); q_hi_all.extend(q_hi.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    return {
        'ef_true': np.array(ef_true_all), 'ef_mean': np.array(ef_mean_all),
        'q_lo': np.array(q_lo_all), 'q_hi': np.array(q_hi_all),
        'cls_true': np.array(cls_true_all), 'cls_prob': np.array(cls_prob_all),
    }

val_results = predict_all(model, val_loader)

rng = np.random.default_rng(42)
n_val = len(val_results['ef_true'])
perm = rng.permutation(n_val)
calib_idx, calibcheck_idx = perm[:n_val // 2], perm[n_val // 2:]

ALPHA = 0.1
E_calib = np.maximum(val_results['q_lo'][calib_idx] - val_results['ef_true'][calib_idx],
                      val_results['ef_true'][calib_idx] - val_results['q_hi'][calib_idx])
Q_hat = np.quantile(E_calib, min(1.0, (1 - ALPHA) * (1 + 1 / len(calib_idx))))
print(f'CQR correction Q_hat: {Q_hat:.3f}')

check_lo = val_results['q_lo'][calibcheck_idx] - Q_hat
check_hi = val_results['q_hi'][calibcheck_idx] + Q_hat
check_covered = ((val_results['ef_true'][calibcheck_idx] >= check_lo) & (val_results['ef_true'][calibcheck_idx] <= check_hi))
print(f'Held-out calibration check coverage (target {(1-ALPHA)*100:.0f}%): {check_covered.mean()*100:.1f}%')
print(f'Mean interval width on this check split: {(check_hi - check_lo).mean():.2f} EF percentage points')
print('(compare to notebook 10 Gaussian-conformal width of ~32.8 points at the same target coverage)')

## HFrEF guaranteed-accuracy curve (unchanged method from notebook 10, recomputed for this model)

In [ ]:
def clopper_pearson_lower(successes, n, confidence=0.95):
    if n == 0:
        return float('nan')
    if successes == 0:
        return 0.0
    return stats.beta.ppf((1 - confidence), successes, n - successes + 1)

coverages = np.arange(1.0, 0.39, -0.05)
eps = 1e-8
cls_entropy_val = -(val_results['cls_prob'] * np.log(val_results['cls_prob'] + eps) +
                     (1 - val_results['cls_prob']) * np.log(1 - val_results['cls_prob'] + eps))
order_calib = calib_idx[np.argsort(cls_entropy_val[calib_idx])]

cls_rows = []
for cov in coverages:
    k = max(10, int(len(order_calib) * cov))
    idx = order_calib[:k]
    preds = (val_results['cls_prob'][idx] > 0.5).astype(int)
    correct = (preds == val_results['cls_true'][idx].astype(int)).sum()
    cls_rows.append({'coverage': cov, 'n': k, 'point_accuracy': correct / k,
                      'guaranteed_lower_bound_95pct': clopper_pearson_lower(correct, k)})
print(pd.DataFrame(cls_rows))

## THE FINAL, ONE-TIME TEST EVALUATION

This is the last time this project touches the test set. The resulting numbers are the ones reported as the paper's primary result.

In [ ]:
test_ds = EchoFrameDataset(test_df, FRAME_CACHE_DIR, filename_col, ef_col, train=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
test_results = predict_all(model, test_loader)

point_mae = mean_absolute_error(test_results['ef_true'], test_results['ef_mean'])
point_auc = roc_auc_score(test_results['cls_true'], test_results['cls_prob'])

test_lo = test_results['q_lo'] - Q_hat
test_hi = test_results['q_hi'] + Q_hat
test_covered = (test_results['ef_true'] >= test_lo) & (test_results['ef_true'] <= test_hi)
test_coverage = test_covered.mean()
test_mean_width = (test_hi - test_lo).mean()

print(f'TEST EF MAE: {point_mae:.3f}')
print(f'TEST HFrEF AUC: {point_auc:.3f}')
print(f'TEST CQR interval coverage (target {(1-ALPHA)*100:.0f}%): {test_coverage*100:.1f}%')
print(f'TEST mean CQR interval width: {test_mean_width:.2f} EF percentage points')

N_BOOTSTRAP = 1000
rng2 = np.random.default_rng(123)
n_test = len(test_results['ef_true'])
boot_maes, boot_aucs = [], []
for _ in range(N_BOOTSTRAP):
    idx = rng2.integers(0, n_test, size=n_test)
    boot_maes.append(mean_absolute_error(test_results['ef_true'][idx], test_results['ef_mean'][idx]))
    try:
        boot_aucs.append(roc_auc_score(test_results['cls_true'][idx], test_results['cls_prob'][idx]))
    except ValueError:
        pass
mae_ci = (np.percentile(boot_maes, 2.5), np.percentile(boot_maes, 97.5))
auc_ci = (np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5))
print(f'\nTEST EF MAE: {point_mae:.3f} (95% CI: {mae_ci[0]:.3f}-{mae_ci[1]:.3f})')
print(f'TEST HFrEF AUC: {point_auc:.3f} (95% CI: {auc_ci[0]:.3f}-{auc_ci[1]:.3f})')

final_cqr_results = {
    'test_mae': point_mae, 'test_mae_ci': mae_ci, 'test_auc': point_auc, 'test_auc_ci': auc_ci,
    'cqr_target_coverage': 1 - ALPHA, 'cqr_empirical_test_coverage': float(test_coverage),
    'cqr_mean_interval_width': float(test_mean_width),
}
with open(os.path.join(PROJECT_ROOT, 'logs', 'final_cqr_test_results.json'), 'w') as f:
    json.dump(final_cqr_results, f, indent=2)
print('\nSaved to logs/final_cqr_test_results.json -- THIS IS THE FINAL REPORTED RESULT.')

In [ ]:
import matplotlib.pyplot as plt

order = np.argsort(test_results['ef_mean'])
widths = test_hi - test_lo

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(test_results['ef_mean'][order], widths[order], s=8, alpha=0.4)
axes[0].set_xlabel('Predicted EF'); axes[0].set_ylabel('CQR interval width')
axes[0].set_title('Interval width vs. predicted EF (adaptivity check)')

sample_idx = np.random.default_rng(1).choice(len(test_results['ef_true']), size=60, replace=False)
sample_idx = sample_idx[np.argsort(test_results['ef_true'][sample_idx])]
axes[1].fill_between(range(len(sample_idx)), test_lo[sample_idx], test_hi[sample_idx], alpha=0.3, label='90% CQR interval')
axes[1].scatter(range(len(sample_idx)), test_results['ef_true'][sample_idx], s=12, color='black', label='True EF')
axes[1].set_xlabel('Test videos (sorted by true EF, random subset of 60)'); axes[1].set_ylabel('EF')
axes[1].set_title('CQR intervals vs. ground truth (sample)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'figures', 'cqr_intervals_final.png'), dpi=150)
plt.show()

## Summary

This is the definitive final model and result for the paper:
- **Point estimates**: EF MAE and HFrEF AUC with bootstrap 95% CIs.
- **EF uncertainty**: CQR intervals with a finite-sample coverage guarantee, tighter and more adaptive than the earlier Gaussian-conformal approach (compare the printed widths directly).
- **HFrEF selective prediction**: Clopper-Pearson guaranteed-accuracy lower bound, cleanly monotonic across coverage levels.
- **No further test-set evaluations should be performed for this project.** Any further methodological changes should be validated on VAL only and reported as such.